In [2]:
from langchain_groq import ChatGroq

In [ ]:
llm = ChatGroq(
    temperature=0, 
    groq_api_key='your_api_key', 
    model_name="meta-llama/llama-4-scout-17b-16e-instruct"
)
response = llm.invoke("how are you? ...")
print(response.content)

I'm doing well, thank you for asking! I'm a large language model, so I don't have feelings or emotions like humans do, but I'm functioning properly and ready to help with any questions or tasks you may have. How about you? How's your day going?


In [4]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://careers.nike.com/software-engineer-ii-itc/job/R-55391")
page_data = loader.load().pop().page_content
print(page_data)

USER_AGENT environment variable not set, consider setting it to identify your requests.























SOFTWARE ENGINEER II, ITC










































Skip to main content
Open Virtual Assistant










Home


Career Areas


Total Rewards


Life@Nike


Purpose










Language





Select a Language

  Deutsch  
  English  
  Español (España)  
  Español (América Latina)  
  Français  
  Italiano  
  Nederlands  
  Polski  
  Tiếng Việt  
  Türkçe  
  简体中文  
  繁體中文  
  עִברִית  
  한국어  
  日本語  








Careers


















Close Menu







Careers






Chat






                                Home
                            



                                Career Areas
                            



                                Total Rewards
                            



                                Life@Nike
                            



                                Purpose
                            










Jordan Careers







Converse Careers










Language











Menu



Return to Previous Menu





In [5]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the 
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):    
        """
)

chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
type(res.content)

str

In [6]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'SOFTWARE ENGINEER II, ITC',
 'experience': ['4+ years of experience in large-scale production-grade software development & platform engineering',
  '4+ years of hands-on experience with AWS or Azure or GCP',
  '2+ years of experience developing data & analytics solutions using Airflow, Spark, NiFi, Kafka',
  '2+ years of experience with ELK stack observability & monitoring tools like SignalFx, NewRelic etc.',
  '2+ years of experience with DevOps stack: GitHub, Jenkins, Docker, Terraform, EKS etc.'],
 'skills': ['Python',
  'Java',
  'Scala',
  'Node.js',
  'Airflow',
  'Spark',
  'NiFi',
  'Kafka',
  'ELK stack',
  'SignalFx',
  'NewRelic',
  'GitHub',
  'Jenkins',
  'Docker',
  'Terraform',
  'EKS',
  'OAuth2.0',
  'OpenID Connect',
  'JWT',
  'Agile',
  'Test-driven development'],
 'description': 'We are looking for a Senior Software Engineer, EADP who excels in team environments and are excited about building cloud native platforms that can scale with the demand of our bu

In [7]:
type(json_res)

dict

In [8]:
import pandas as pd

df = pd.read_csv("my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the techstack data
vectors = vectorizer.fit_transform(df['Techstack'])

def get_similar_links(query_skills, n_results=2):
    # Transform the query
    query_vec = vectorizer.transform([query_skills])
    
    # Calculate similarities
    similarities = cosine_similarity(query_vec, vectors)
    
    # Get top matches
    top_indices = similarities[0].argsort()[-n_results:][::-1]
    
    # Return the links
    return [{"links": df.iloc[idx]['Links']} for idx in top_indices]

In [10]:
job = json_res
job['skills']

['Python',
 'Java',
 'Scala',
 'Node.js',
 'Airflow',
 'Spark',
 'NiFi',
 'Kafka',
 'ELK stack',
 'SignalFx',
 'NewRelic',
 'GitHub',
 'Jenkins',
 'Docker',
 'Terraform',
 'EKS',
 'OAuth2.0',
 'OpenID Connect',
 'JWT',
 'Agile',
 'Test-driven development']

In [11]:
# Extract job information and parse JSON
chain_extract = prompt_extract | llm 
res = chain_extract.invoke(input={'page_data':page_data})
json_parser = JsonOutputParser()
jobs = json_parser.parse(res.content)

# Convert to list if single job
if not isinstance(jobs, list):
    jobs = [jobs]

# Process each job
for job in jobs:
    # Convert skills to string if it's a list
    skills = job['skills']
    if isinstance(skills, list):
        skills = ', '.join(skills)
    
    # Get similar links
    links = get_similar_links(skills)
    
    # Generate email
    prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Saikat, a business development executive at XYZ COMPANY. XYZ COMPANY is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of XYZ COMPANY 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase XYZ COMPANY portfolio: {link_list}
        Remember you are Saikat, BDE at XYZ COMPANY. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        """
    )

    chain_email = prompt_email | llm
    res = chain_email.invoke({"job_description": str(job), "link_list": links})
    print(res.content)
    print("\n---\n")  # Separator between multiple emails

Subject: Expert Software Engineers for Your Large-Scale Production-Grade Software Development Needs

Dear Hiring Manager,

I came across the job posting for a Software Engineer II, ITC at your organization, and I was impressed by the scale and complexity of the projects you're working on. At XYZ COMPANY, we specialize in providing AI and software consulting services that cater to the unique needs of enterprises like yours.

With over [number] years of experience in delivering tailored solutions, we've developed a strong expertise in large-scale production-grade software development, cloud-native platforms, and DevOps. Our team of experts has extensive experience with AWS, Azure, GCP, and a range of tools including Airflow, Spark, NiFi, Kafka, ELK stack, and more.

I'd like to highlight that our company has a proven track record in:

* Developing scalable and efficient software solutions using Python, Java, Scala, Node.js, and other programming languages.
* Designing and implementing cl